# Deploy electricity bill model — run guide

Pipeline: checkpoint on GCS is merged with the base model via Cloud Build, the merged model is baked into a Docker image at build time, then the image is deployed to Cloud Run.

Run these commands in Cloud Shell, not in PowerShell. Each cell prints the exact command to run; the actual execution line is commented out so you can copy the command into Cloud Shell instead of running it from the notebook's own environment.

In [1]:
!gcloud config list

[accessibility]
screen_reader = False
[core]
account = ngocchiien23@gmail.com
disable_usage_reporting = True
project = first-orc-chien



Your active configuration is: [default]


## Configuration

In [8]:
import json

try:
    with open("gcs_config.json") as f:
        cfg = json.load(f)
    PROJECT_ID = cfg["project_id"]
    REGION = cfg["region"]
except Exception as e:
    print(f"Error loading gcs_config.json: {str(e)}")
    PROJECT_ID = "first-orc-chien"
    REGION = "asia-southeast1"

REPO = f"asia-southeast1-docker.pkg.dev/{PROJECT_ID}/swift-json-vlm-container-finetuned"
MERGE_JOB_DIR = "merge-job"
DEPLOY_DIR = "deployments"

SERVICE = "vllm-serve-bill"
CHECKPOINT_GCS = "gs://electric-bill-dataset-gcs/output/model/v0-20260623-161620/checkpoint-350"
MERGED_GCS = "gs://electric-bill-dataset-gcs/output/merged/v0-20260623-161620"
MODEL_NAME = "document-to-json"
IMAGE_URI = f"{REPO}/inference-bill:latest"
ENV_FILE = "deploy.env.electric-bill"
DEPLOY_SCRIPT = "submit-deploy-bill.sh"

print("Project:", PROJECT_ID)
print("Region:", REGION)
print("Service:", SERVICE)
print("Checkpoint:", CHECKPOINT_GCS or "(none, merge already done)")
print("Merged output:", MERGED_GCS)
print("Served name:", MODEL_NAME)
print("Image:", IMAGE_URI)


Project: first-orc-chien
Region: asia-southeast1
Service: vllm-serve-bill
Checkpoint: gs://electric-bill-dataset-gcs/output/model/v0-20260623-161620/checkpoint-350
Merged output: gs://electric-bill-dataset-gcs/output/merged/v0-20260623-161620
Served name: document-to-json
Image: asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned/inference-bill:latest


## 1. Merge LoRA into the base model

Runs as a Cloud Build job. Skip this if the merged model already exists on GCS.

In [9]:
cmd = (
    f"gcloud builds submit --config={MERGE_JOB_DIR}/cloudbuild-merge.yaml "
    f"--project={PROJECT_ID} "
    f"--substitutions=_CHECKPOINT_GCS_PATH={CHECKPOINT_GCS},"
    f"_MERGED_OUTPUT_GCS_PATH={MERGED_GCS} "
    f"{MERGE_JOB_DIR}"
)
print(cmd)
# !{cmd}


gcloud builds submit --config=merge-job/cloudbuild-merge.yaml --project=first-orc-chien --substitutions=_CHECKPOINT_GCS_PATH=gs://electric-bill-dataset-gcs/output/model/v0-20260623-161620/checkpoint-350,_MERGED_OUTPUT_GCS_PATH=gs://electric-bill-dataset-gcs/output/merged/v0-20260623-161620 merge-job


Confirm the merged model exists on GCS before building the image.

In [10]:
cmd = f"gsutil ls {MERGED_GCS}/"
print(cmd)
# !{cmd}


gsutil ls gs://electric-bill-dataset-gcs/output/merged/v0-20260623-161620/


## 2. Build the image

The merged model is baked into the image at build time via `MODEL_GCS_PATH`. Cold start loads the model from local disk, no GCS download at runtime. Rebuild only when the model or `Dockerfile`/`entrypoint.sh` changes.

In [11]:
import os

cmd = (
    f"gcloud builds submit --config={DEPLOY_DIR}/cloudbuild-bake.yaml "
    f"--project={PROJECT_ID} "
    f"--substitutions=_MODEL_PATH={MERGED_GCS},_IMAGE_TAG={IMAGE_URI} "
    f"{DEPLOY_DIR}"
)
print(cmd)
# !{cmd}

gcloud builds submit --config=deployments/cloudbuild-bake.yaml --project=first-orc-chien --substitutions=_MODEL_PATH=gs://electric-bill-dataset-gcs/output/merged/v0-20260623-161620,_IMAGE_TAG=asia-southeast1-docker.pkg.dev/first-orc-chien/swift-json-vlm-container-finetuned/inference-bill:latest deployments


## 3. Deploy to Cloud Run

1 NVIDIA L4 GPU, 8 vCPU, 32GB memory, `--min-instances 0`. The model is already inside the image, so `entrypoint.sh` no longer downloads from GCS.

In [12]:
cmd = f"gcloud compute regions describe {REGION} --project {PROJECT_ID}"
print(cmd)
# !{cmd}


gcloud compute regions describe asia-southeast1 --project first-orc-chien


In [13]:
cmd = f"bash {DEPLOY_DIR}/{DEPLOY_SCRIPT}"
print(cmd)
# !{cmd}


bash deployments/submit-deploy-bill.sh


## 4. Allow public calls

Grants `roles/run.invoker` to `allUsers`. Fine for testing, revisit before production traffic.

In [14]:
cmd = (
    f"gcloud run services add-iam-policy-binding {SERVICE} "
    f"--region={REGION} --member=allUsers --role=roles/run.invoker "
    f"--project={PROJECT_ID}"
)
print(cmd)
# !{cmd}


gcloud run services add-iam-policy-binding vllm-serve-bill --region=asia-southeast1 --member=allUsers --role=roles/run.invoker --project=first-orc-chien


## 5. Get the service URL

In [15]:
cmd = (
    f"gcloud run services describe {SERVICE} --region {REGION} "
    f"--project {PROJECT_ID} --format 'value(status.url)'"
)
print(cmd)
# !{cmd}


gcloud run services describe vllm-serve-bill --region asia-southeast1 --project first-orc-chien --format 'value(status.url)'


## 6. Test with curl

Replace `(SERVICE_URL)` with the URL from the previous step.

In [16]:
print(f"curl -i -H \"Authorization: Bearer $(gcloud auth print-identity-token)\" (SERVICE_URL)/health")


curl -i -H "Authorization: Bearer $(gcloud auth print-identity-token)" (SERVICE_URL)/health


## 7. Cleanup

In [ ]:
!gcloud run services delete {SERVICE} --region {REGION} --project {PROJECT_ID} --quiet


In [ ]:
!gcloud run services update {SERVICE} --region={REGION} --max-instances=0 --project={PROJECT_ID}


In [ ]:
!gcloud run services list --region={REGION} --project={PROJECT_ID}


With `--min-instances 0` set at deploy time, the service scales to zero automatically after idle traffic. The commands above force that immediately or remove the service.

## Troubleshooting

Run in Cloud Shell, not PowerShell, since it parses quotes and commas differently and breaks `--set-env-vars`/`--startup-probe`. Rebuild the image whenever the model changes, since the model is now baked into the image rather than downloaded at runtime. If the build fails on `gsutil cp`, grant `roles/storage.objectViewer` to the Cloud Build service account on the source bucket. Startup probe timeout is already handled with `failureThreshold=60, periodSeconds=10` in the deploy script.